# 0. import packages and read in data

In [24]:
import detectlanguage

detectlanguage.configuration.api_key = "8bfa4f869a49a2ee0ded3ac480da933d"

In [1]:
import pandas as pd
import re
from bs4 import BeautifulSoup
import unicodedata
import contractions

In [2]:
resume_df = pd.read_csv('extracted_cvs.csv', on_bad_lines='warn')
jobs_df = pd.read_csv('job_listings3.csv') #de url links werken, maar je moet ze volledig kopieren

/var/folders/8x/qw4pmxy96y33j71r3sj_v4_00000gn/T/ipykernel_4353/961090222.py:1: ParserWarning: Skipping line 8: expected 2 fields, saw 3

  resume_df = pd.read_csv('extracted_cvs.csv', on_bad_lines='warn')


In [3]:
resume_df

,file_path,content
0,./CV Mikail.pdf,CURRICULUM VITAE\n\nMIKAIL MALCIKAN\n\nL I N K...
1,Ayush_Shrestha_CV.pdf,AYUSH SHRESTHA\n\nS t u d e n t\n\nABOUT ME\n\...
2,Cv_George_Lemmens.pdf,George Lemmens\n\nErvaring\n\n2020 - 2021\n\nM...
3,CV Mikail.pdf,CURRICULUM VITAE\n\nMIKAIL MALCIKAN\n\nL I N K...
4,Ayush_Shrestha_CV.pdf,AYUSH SHRESTHA\n\nS t u d e n t\n\nABOUT ME\n\...
5,Ayush_Shrestha_CV.pdf,AYUSH SHRESTHA\n\nS t u d e n t\n\nABOUT ME\n\...


In [4]:
jobs_df.head()

,Title,Address,Link,Job Description
0,Data Scientist,"Brussels, Brussels Region, Belgium",https://be.linkedin.com/jobs/view/data-scienti...,Are you looking to make a measurable impact fo...
1,Junior Data Scientist,"Ixelles, Belgium",https://be.linkedin.com/jobs/view/junior-data-...,Direct message the job poster from Pauwels Con...
2,Data Analyst,"Brussels, Brussels Region, Belgium",https://be.linkedin.com/jobs/view/data-analyst...,Direct message the job poster from Nemeon Mark...
3,"(Junior) Data, Insights & Analytics Consultant","Kortrijk, Flemish Region, Belgium",https://be.linkedin.com/jobs/view/junior-data-...,Do you want to become an expert in Data Analyt...
4,Data Scientist,"Brussels, Brussels Region, Belgium",https://be.linkedin.com/jobs/view/data-scienti...,Direct message the job poster from EUROPEAN DY...


In [5]:
jobs_df.isna().mean()

Title              0.0
Address            0.0
Link               0.0
Job Description    0.0
dtype: float64

In [6]:
jobs_df.dropna(inplace=True)

# 1.text cleaning

In [7]:
def cleaning(text):
    if not isinstance(text, str):
        return ""
    
    # Normalize Unicode characters
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
    
    # Expand contractions
    #text = contractions.fix(text)
    
    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()
    
    # Remove @mentions
    text = re.sub(r"@\w+", "", text)
    
    # Remove URLs
    text = re.sub(r"https?://\S+", "", text)
    
    # Remove hashtags
    text = re.sub(r"#\w+", "", text)
    
    # Remove non-ASCII characters
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    
    # Remove newline characters
    text = re.sub(r"\n", " ", text)
    
    # Remove leading/trailing spaces
    text = re.sub(r"^\s+", "", text)
    text = re.sub(r"\s+$", "", text)
    
    # Remove multiple spaces
    text = re.sub(r"[ |\t]+", " ", text)
    
    # Remove parentheses and slashes
    text = re.sub(r"[()]", "", text)
    text = re.sub(r"/", " ", text)
    
    # Remove commas and periods
    text = re.sub(r",", " ", text)
    text = re.sub(r"\. ", " ", text)
    
    # Remove colons and semicolons
    text = re.sub(r":", " ", text)
    text = re.sub(r";", " ", text)
    
    # Remove numbers
    text = re.sub(r"\b\d+\b", "", text)
    
    # Remove carriage returns
    text = re.sub(r"\r", "", text)
    
    # Remove quotes and ampersands
    text = re.sub(r"\"", "", text)
    text = re.sub(r"&", "", text)
    
    # Remove hyphens, plus signs, asterisks, and equal signs
    text = re.sub(r"-", "", text)
    text = re.sub(r"\+", "", text)
    text = re.sub(r"\*", "", text)
    text = re.sub(r"\.", "", text)
    text = re.sub(r"=", "", text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters (if any left)
    text = re.sub(r"[^\w\s]", "", text)
    
    # Remove multiple spaces again just in case
    text = re.sub(r"\s+", " ", text)
    
    return text


In [8]:
# Clean and translate the 'content' column
resume_df['clean_content'] = resume_df['content'].apply(lambda x: cleaning(x))

# Clean and translate the 'Job Description' column
jobs_df['clean_description'] = jobs_df['Job Description'].apply(lambda x: cleaning(str(x)))
jobs_df['clean_address'] = jobs_df['Address'].apply(lambda x: cleaning(str(x)))


In [9]:
# Temporarily change the display options
#check cleaning function
'''with pd.option_context('display.max_colwidth', None):
    print('cleaned:',resume_df[['clean_content']])
    print('content:',resume_df[['content']])'''

'''with pd.option_context('display.max_colwidth', None):
    print('cleaned:',jobs_df['clean_description'].head())
    print('content:',jobs_df['Job Description'].head())'''

with pd.option_context('display.max_colwidth', None):
    print('cleaned:',jobs_df['clean_address'].head())
    print('content:',jobs_df['Address'].head())

cleaned: 0    brussels brussels region belgium
1                     ixelles belgium
2    brussels brussels region belgium
3     kortrijk flemish region belgium
4    brussels brussels region belgium
Name: clean_address, dtype: object
content: 0    Brussels, Brussels Region, Belgium
1                      Ixelles, Belgium
2    Brussels, Brussels Region, Belgium
3     Kortrijk, Flemish Region, Belgium
4    Brussels, Brussels Region, Belgium
Name: Address, dtype: object


In [39]:
'''# Function to detect language and handle empty text
def detect_language(text):
    if text:
        result = detectlanguage.detect(text)
        if result and isinstance(result, list) and isinstance(result[0], dict):
            return result[0]['language']
        else:
            return 'Unknown'
    else:
        return 'Empty'

# Apply the language detection function
jobs_df['clean_description_language'] = jobs_df['clean_description'].apply(detect_language)
resume_df['clean_content_language'] = resume_df['clean_content'].apply(detect_language)
'''

"# Function to detect language and handle empty text\ndef detect_language(text):\n    if text:\n        result = detectlanguage.detect(text)\n        if result and isinstance(result, list) and isinstance(result[0], dict):\n            return result[0]['language']\n        else:\n            return 'Unknown'\n    else:\n        return 'Empty'\n\n# Apply the language detection function\njobs_df['clean_description_language'] = jobs_df['clean_description'].apply(detect_language)\nresume_df['clean_content_language'] = resume_df['clean_content'].apply(detect_language)\n"

In [21]:
###
from deep_translator import GoogleTranslator

# Define a function to translate text
def translate_text(text):
    if text is None or text == '':
        return ''
    else:
        try:
            translation = GoogleTranslator(source='auto', target='en').translate(text)
            return translation
        except Exception as e:
            print(f'Error translating text: {e}')
            return

# Apply the function to the clean_description column
jobs_df['translated_description'] = jobs_df['clean_description'].apply(lambda x: translate_text(x) if x else 'Empty')

# Apply the function to the clean_address column #niet nodig
#jobs_df['translated_address'] = jobs_df['clean_address'].apply(lambda x: translate_text(x) if x else 'Empty')

# Apply the function to the clean_content column
resume_df['translated_content'] = resume_df['clean_content'].apply(lambda x: translate_text(x) if x else 'Empty')

Error translating text: are you looking to make a measurable impact for your clients with data science ready to expand your skillset in different tools methodologies and across various industries do you have an innate passion for business and entrepreneurial sense while having an interest in the technical aspects of the job what is your mission help innovative companies thrive in turning their objectives into results through smarter use of data more concretely what will you do daytoday as a data scientist you will play a vital role in the endtoend delivery of our projects therefore you will work closely with stakeholders to prioritize business requirements manage data collection analyze results provide reports model data interpret trends and patterns communicate the results and next steps effectively with clients our assignments include data audit data preparation descriptive analysis modeling presenting the results senior data scientists supervise and coach junior data scientists in d

KeyboardInterrupt: 

In [40]:
# Temporarily change the display options
#check translation function
'''with pd.option_context('display.max_colwidth', None):
    print('cleaned:',resume_df[['clean_content']])
    print('translated:',resume_df[['translated_content']])'''

with pd.option_context('display.max_colwidth', None):
    print('cleaned:',jobs_df['clean_description'].head())
    print('translated:',jobs_df['translated_description'].head())

'''with pd.option_context('display.max_colwidth', None):
    print('cleaned:',jobs_df['clean_address'].head())
    print('translated:',jobs_df['translated_address'].head())'''

cleaned: 0    are you looking to make a measurable impact for your clients with data science ready to expand your skillset in different tools methodologies and across various industries do you have an innate passion for business and entrepreneurial sense while having an interest in the technical aspects of the job what is your mission help innovative companies thrive in turning their objectives into results through smarter use of data more concretely what will you do daytoday as a data scientist you will play a vital role in the endtoend delivery of our projects therefore you will work closely with stakeholders to prioritize business requirements manage data collection analyze results provide reports model data interpret trends and patterns communicate the results and next steps effectively with clients our assignments include data audit data preparation descriptive analysis modeling presenting the results senior data scientists supervise and coach junior data scientists in delivering 

"with pd.option_context('display.max_colwidth', None):\n    print('cleaned:',jobs_df['clean_address'].head())\n    print('translated:',jobs_df['translated_address'].head())"

# 2.similarity matching

In [ ]:
#pip install spacy

#import spacy.cli
#spacy.cli.download("en_core_web_md") #(this is the medium size model 40mb)
#or
#spacy.cli.download("en_core_web_lg") (this is the large model 700mb) we should maybe switch at the end to increase model performance

In [ ]:
import spacy

In [ ]:
nlp = spacy.load("en_core_web_md")

### testing

In [ ]:
j1 = nlp(jobs_df['clean_description'][0])
j1 

are you looking to make a measurable impact for your clients with data science? ready to expand your skillset in different tools  methodologies  and across various industries? do you have an innate passion for business and entrepreneurial sense while having an interest in the technical aspects of the job? what is your mission? help innovative companies thrive in turning their objectives into results  through smarter use of data more concretely  what will you do daytoday? as a data scientist  you will play a vital role in the endtoend delivery of our projects therefore  you will work closely with stakeholders to  prioritize business requirements manage data collection analyze results provide reports model data interpret trends and patterns communicate the results and next steps effectively with clients our assignments include  data audit data preparation descriptive analysis modeling presenting the results senior data scientists supervise and coach junior data scientists in delivering t

In [ ]:
r1 = nlp(resume_df['clean_content'][0])
r1

curriculum vitae mikail malcikan l i n k e d i n personal profile motivated business engineering master's student with strong analytical and problemsolving skills adaptable and dedicated  i excel in collaborative team environments eager to contribute academic and practical expertise to drive innovative solutions for organizational success talents and skills flexible and stressresistant it knowledge  ms office java r sql python pyspark spss sap contact gsm      email  mikailmalcikancom gaverstraat    wondelgem education universiteit gent master of science in business engineering  data analytics skills  pyspark sql data analysis python r modeling and simulation related  studentenvereniging flux bestuurslid  penningmeester winner of kaggle competition room price prediction dataset machine learning atheneum wispelberg aso  economie wiskunde experiences studentjobs claimhandler  gates event worker  sevent orderpicker  dsv solutions sorter  bpost orderpicker  barvatar delivery associate  alb

In [ ]:
r2 = nlp(resume_df['clean_content'][2])
r2

george lemmens ervaring    mowi en bakkerij notredame inpakken  logistiek en verkoop  burger king en bakkerij notredame keuken en verkoop  az zeno schoonmaak  albert heijn rekkenvuller contact telefoon  email lemmensgeorgecom adres rond den heerdstraat    brugge educatie  latijnwiskunde sintjozef humaniora heden burgerlijk ingenieur afstudeerrichting biomedische ingenieurstechnieken universiteit gent skills microsoft word microsoft powerpoint microsoft excel python maple talen nederlands  moedertaal engels  vlot frans  basis

In [ ]:
j1.similarity(r1)

0.905266056330655

In [ ]:
j1.similarity(r2)

0.3258069352590815

In [ ]:
j1_nouns = " ".join([token.lemma_ for token in j1 if token.pos_ == 'NOUN'])
r1_nouns = " ".join([token.lemma_ for token in r1 if token.pos_ == 'NOUN'])
r2_nouns = " ".join([token.lemma_ for token in r2 if token.pos_ == 'NOUN'])


j1_verbs = " ".join([token.lemma_ for token in j1 if token.pos_ == 'VERB'])
r1_verbs = " ".join([token.lemma_ for token in r1 if token.pos_ == 'VERB'])
r2_verbs = " ".join([token.lemma_ for token in r2 if token.pos_ == 'VERB'])


j1_adjs = " ".join([token.lemma_ for token in j1 if token.pos_ == 'ADJ'])
r1_adjs = " ".join([token.lemma_ for token in r1 if token.pos_ == 'ADJ'])
r2_adjs = " ".join([token.lemma_ for token in r2 if token.pos_ == 'ADJ'])

In [ ]:
print(nlp(j1_nouns).similarity(nlp(r1_nouns)))
print(nlp(j1_nouns).similarity(nlp(r2_nouns)))

0.9528472237395565
0.6108765039901467


In [ ]:
print(nlp(j1_verbs).similarity(nlp(r1_verbs)))
print(nlp(j1_verbs).similarity(nlp(r2_verbs)))

0.8680568332617608
0.5007512495849951


In [ ]:
print(nlp(j1_adjs).similarity(nlp(r1_adjs)))
print(nlp(j1_adjs).similarity(nlp(r2_adjs)))

0.9196338919736171
0.0


/var/folders/8x/qw4pmxy96y33j71r3sj_v4_00000gn/T/ipykernel_1825/955577902.py:2: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  print(nlp(j1_adjs).similarity(nlp(r2_adjs)))


### trying

In [ ]:
i = 1

In [ ]:
resume_df['clean_content'][i]

"ayush shrestha s t u d e n t about me  ayush1214shresthacom sh1214shrestha  as a future graduate from the business engineering in data science program  my passion lies in the fields of data science  machine learning  and beyond i am deeply intrigued by the power of datadriven insights to transform businesses and am eager to contribute my skills and knowledge to this dynamic field torhoutse steenweg    sintandries experiences rijbewijs b education oct    bijleshuis private tutor msc business engineering  data analytics ghent university    programming languages python r java sql languages english french dutch at bijleshuis  i served as a private tutor  providing tailored schooling to students spanning various age groups in subjects such as economics  mathematics  physics  and more it was a gratifying experience to help students achieve academic success and build confidence in their abilities across diverse subjects jun   jul  pittman seafoods administrative assistant and receptionist at

In [ ]:
# Tokenize the resume content (assumes one resume per DataFrame row)
resume_doc = nlp(resume_df['clean_content'][i])

# Create a new column in jobs_df to hold similarity scores
jobs_df['similarity_measure'] = 0.0  # Initialize with default value

# Calculate the similarity of the resume with each job description
for idx, row in jobs_df.iterrows():
    job_doc = nlp(row['clean_description'])
    similarity = resume_doc.similarity(job_doc)  # Calculate similarity
    jobs_df.at[idx, 'similarity_measure'] = similarity  # Assign similarity to the DataFrame

# Display the DataFrame with the added similarity measure column
jobs_df=jobs_df.sort_values('similarity_measure', ascending=False)

In [ ]:
jobs_df.head()

,Title,Address,Link,Job Description,clean_description,clean_address,translated_description,translated_address,similarity_measure
47,Research Scientist,"Leuven, Flemish Region, Belgium",https://be.linkedin.com/jobs/view/research-sci...,ArtiQ (a Clario Company) is seeking a Research...,artiq a clario company is seeking a research s...,leuven flemish region belgium,,,0.980858
25,Data Analyst,"Antwerp, Flemish Region, Belgium",https://be.linkedin.com/jobs/view/data-analyst...,Functieomschrijving The data analysts of each...,functieomschrijving the data analysts of each ...,antwerp flemish region belgium,,,0.980852
14,Data scientist,"Antwerp, Flemish Region, Belgium",https://be.linkedin.com/jobs/view/data-scienti...,Functieomschrijving The data analysts in each...,functieomschrijving the data analysts in each ...,antwerp flemish region belgium,,,0.980833
1,Junior Data Scientist,"Ixelles, Belgium",https://be.linkedin.com/jobs/view/junior-data-...,Direct message the job poster from Pauwels Con...,direct message the job poster from pauwels con...,ixelles belgium,,,0.980016
19,Data Scientist,"Brussels, Brussels Region, Belgium",https://be.linkedin.com/jobs/view/data-scienti...,Direct message the job poster from Exsolvæ Hug...,direct message the job poster from exsolv hugo...,brussels brussels region belgium,,,0.979827


In [ ]:
jobs_df.tail()

,Title,Address,Link,Job Description,clean_description,clean_address,translated_description,translated_address,similarity_measure
48,Bmatix - Junior Data Engineer,"Kontich, Flemish Region, Belgium",https://be.linkedin.com/jobs/view/bmatix-junio...,Who We Are Bmatix specialiseert zich in Busin...,who we are bmatix specialiseert zich in busine...,kontich flemish region belgium,,,0.432452
49,Elektricien,"Zwijndrecht, Flemish Region, Belgium",https://be.linkedin.com/jobs/view/elektricien-...,"Cegelec, een merk van VINCI Energies, is gespe...",cegelec een merk van vinci energies is gespe...,zwijndrecht flemish region belgium,,,0.409590
34,Data Analyst,Antwerp Metropolitan Area,https://be.linkedin.com/jobs/view/data-analyst...,Direct message the job poster from Cartière Ro...,direct message the job poster from cartire rob...,antwerp metropolitan area,,,0.280198
42,Data Analyst,"Brussels, Brussels Region, Belgium",https://be.linkedin.com/jobs/view/data-analyst...,"Pour un de nos client, acteur bancaire, nous s...",pour un de nos client acteur bancaire nous s...,brussels brussels region belgium,,,0.266670
36,Junior Data Analyst,"Flemish Brabant, Flemish Region, Belgium",https://be.linkedin.com/jobs/view/junior-data-...,Direct message the job poster from Cartière Ma...,direct message the job poster from cartire mar...,flemish brabant flemish region belgium,,,0.241206
